In [5]:
import asyncio
import nest_asyncio
import time

# This library patches asyncio to allow nested event loops
nest_asyncio.apply()

async def inner_async():
    print("*** inner function executed ***")
    await asyncio.sleep(1)
    return 0.123

def inner_sync():
    return asyncio.get_running_loop().run_until_complete(inner_async())
    
async def test_async():
    result = inner_sync()
    print(f"*** test executed {result} ***")

In [6]:
import ast
import sys
import traceback

async def execute_resilient(code_str: str, global_context: dict = None, local_context: dict = None) -> None:
    if global_context is None:
        global_context = globals()
    if local_context is None:
        local_context = global_context

    tree = ast.parse(code_str)
    for node in tree.body:
        wrapper = ast.Module(body=[node], type_ignores=[])
        ast.fix_missing_locations(wrapper)
        
        try:
            code_obj = compile(wrapper, filename="<string>", mode="exec")
            exec(code_obj, global_context, local_context)
        except Exception:
            continue

In [10]:
start = time.perf_counter()
code = "result = inner_sync()"
local_context = {}
tasks = [execute_resilient(code, None, local_context) for _ in range(25)]
results = await asyncio.gather(*tasks)
print(f"Duration {time.perf_counter() - start}")

*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
*** inner function executed ***
Duration 1.004080612998223


In [11]:
print(local_context)

{'result': 0.123}
